# Data Cleaning, Table Integration and Executive EDA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmavigo/retail-demand-forecasting/blob/main/notebooks/02_data_cleaning_and_eda.ipynb)

This notebook documents the complete path from three raw Kaggle tables to clean business metrics and five-year visual analysis.

## 1. Environment and official data

In [ ]:
import sys, subprocess
from pathlib import Path

if 'google.colab' in sys.modules:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'kagglehub>=1.0,<2', 'pandas>=2.2,<3', 'numpy>=2,<3',
        'plotly>=5.24,<7', 'scikit-learn>=1.5,<2', 'lightgbm>=4.5,<5'
    ])

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
pd.set_option('display.max_columns', 30)


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('m5-forecasting-accuracy')

print("Path to competition files:", path)

DATA_DIR = Path(path)
if not (DATA_DIR / 'calendar.csv').exists():
    DATA_DIR = next(p.parent for p in DATA_DIR.rglob('calendar.csv'))

required = {'calendar.csv', 'sell_prices.csv', 'sales_train_evaluation.csv'}
available = {p.name for p in DATA_DIR.glob('*.csv')}
assert required.issubset(available), f"Missing files: {sorted(required - available)}"


## 2. Load raw tables and validate business keys

In [ ]:
sales = pd.read_csv(DATA_DIR / 'sales_train_evaluation.csv')
calendar = pd.read_csv(DATA_DIR / 'calendar.csv', parse_dates=['date'])
prices = pd.read_csv(DATA_DIR / 'sell_prices.csv')
day_cols = [c for c in sales if c.startswith('d_')]

checks = pd.Series({
    'duplicate_sales_ids': sales.id.duplicated().sum(),
    'duplicate_calendar_days': calendar.d.duplicated().sum(),
    'duplicate_price_keys': prices.duplicated(['store_id', 'item_id', 'wm_yr_wk']).sum(),
    'missing_prices': prices.sell_price.isna().sum(),
    'negative_prices': (prices.sell_price < 0).sum(),
})
display(checks.to_frame('count'))
assert checks.sum() == 0, 'Review failed quality checks before continuing.'


## 3. Explain the relational model
`d` connects demand to calendar. Calendar contributes `wm_yr_wk`, which combines with store and item to identify the correct weekly price.

In [ ]:
id_cols = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
preview = (
    sales.loc[:99, id_cols + day_cols]
    .melt(id_vars=id_cols, var_name='d', value_name='units')
    .merge(calendar, on='d', how='left', validate='many_to_one')
    .merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left', validate='many_to_one')
)
preview['estimated_revenue'] = preview.units * preview.sell_price
display(preview[['date', 'd', 'store_id', 'item_id', 'cat_id', 'units', 'sell_price', 'estimated_revenue']].head(10))
print('Preview rows:', f'{len(preview):,}', '| Missing dates:', preview.date.isna().sum())


## 4. Integrate every product without a 59-million-row melt
The complete calculation pivots weekly price once and processes products in blocks. This preserves the exact join logic while controlling memory.

In [ ]:
calendar_days = calendar.set_index('d').loc[day_cols].reset_index()
units = sales[day_cols].to_numpy(dtype=np.float32)
row_index = pd.MultiIndex.from_frame(sales[['store_id', 'item_id']])
weeks = calendar_days.wm_yr_wk.drop_duplicates().tolist()
price_wide = prices.pivot(index=['store_id', 'item_id'], columns='wm_yr_wk', values='sell_price').reindex(index=row_index, columns=weeks)
week_position = {week: i for i, week in enumerate(weeks)}
day_week_positions = np.array([week_position[w] for w in calendar_days.wm_yr_wk])

daily_revenue = np.zeros(len(day_cols), dtype=np.float64)
row_revenue = np.zeros(len(sales), dtype=np.float64)
missing_price_sales = 0
for start in range(0, len(sales), 1000):
    stop = min(start + 1000, len(sales))
    block_units = units[start:stop]
    block_prices = price_wide.iloc[start:stop].to_numpy(dtype=np.float32)[:, day_week_positions]
    missing_price_sales += int(((block_units > 0) & np.isnan(block_prices)).sum())
    block_revenue = block_units * np.nan_to_num(block_prices, nan=0.0)
    daily_revenue += block_revenue.sum(axis=0)
    row_revenue[start:stop] = block_revenue.sum(axis=1)

print('Positive sales without price:', missing_price_sales)


## 5. Create trusted analytical tables

In [ ]:
daily = calendar_days[['date', 'year', 'month', 'weekday', 'event_name_1', 'event_type_1']].copy()
daily['units'] = units.sum(axis=0)
daily['estimated_revenue'] = daily_revenue
daily['average_selling_price'] = daily.estimated_revenue.div(daily.units).replace([np.inf], np.nan)
daily['moving_average_28'] = daily.units.rolling(28, min_periods=1).mean()

product = sales[['item_id', 'cat_id', 'dept_id']].copy()
product['units'] = units.sum(axis=1)
product['estimated_revenue'] = row_revenue
product = product.groupby(['item_id', 'cat_id', 'dept_id'], as_index=False).sum().sort_values('estimated_revenue', ascending=False)
product['revenue_share'] = product.estimated_revenue / product.estimated_revenue.sum()
product['cumulative_revenue_share'] = product.revenue_share.cumsum()
product['abc_class'] = np.select([product.cumulative_revenue_share <= .80, product.cumulative_revenue_share <= .95], ['A', 'B'], default='C')

store_summary = pd.DataFrame([
    {'store_id': store, 'units': units[sales.store_id.eq(store)].sum(), 'estimated_revenue': row_revenue[sales.store_id.eq(store)].sum()}
    for store in sorted(sales.store_id.unique())
])
category_summary = pd.DataFrame([
    {'cat_id': cat, 'units': units[sales.cat_id.eq(cat)].sum(), 'estimated_revenue': row_revenue[sales.cat_id.eq(cat)].sum()}
    for cat in sorted(sales.cat_id.unique())
])

display(pd.Series({'Units sold': daily.units.sum(), 'Estimated revenue': daily.estimated_revenue.sum(), 'Products': len(product), 'Stores': len(store_summary)}).to_frame('value'))


## 6. Executive visual analysis

In [ ]:
fig = go.Figure([
    go.Scatter(x=daily.date, y=daily.units, name='Daily units', opacity=.25),
    go.Scatter(x=daily.date, y=daily.moving_average_28, name='28-day average', line={'width': 3}),
])
fig.update_layout(title='Five-year demand and trend')
fig.show()
px.bar(store_summary.sort_values('estimated_revenue'), x='estimated_revenue', y='store_id', orientation='h', title='Estimated revenue by store').show()
px.bar(category_summary, x='cat_id', y='estimated_revenue', color='cat_id', title='Estimated revenue by category').show()
p = product.reset_index(drop=True).copy(); p['product_share'] = (p.index + 1) / len(p)
px.line(p, x='product_share', y='cumulative_revenue_share', title='Product revenue Pareto curve').show()
display(product.head(20))


## Limitations
M5 contains aggregated unit demand, not transactions, customers or basket IDs. Revenue is therefore estimated as units multiplied by weekly selling price; true average ticket cannot be calculated.